# The deeperfly pipeline, step by step

This notebook reproduces what **`deeperfly run`** does, but one module at a time,
so every intermediate output is visible and plotted. We run on the synchronized
7-camera recording in `examples/data/` (one MP4 video per camera,
`camera_0.mp4` &hellip; `camera_6.mp4`).

The `run` CLI (`src/deeperfly/cli.py::_cmd_run`) wires together:

1. **`cameras.CameraGroup`** + **`skeleton.Skeleton`** &mdash; the camera rig and what we track.
2. **`pose2d` detector** &mdash; a stacked-hourglass network producing per-joint heatmaps.
3. **`pose2d.inference`** &mdash; preprocess &rarr; heatmaps &rarr; 2D skeletons per camera.
4. **`pipeline.run_from_points2d`** &mdash; visibility masking &rarr; camera calibration
   (bundle adjustment) &rarr; robust triangulation (RANSAC multi-view consensus) &rarr;
   temporal smoothing &rarr; a saved **`io.PoseResult`**.

We unfold steps 1&ndash;4 below and visualize each stage with matplotlib. The final cell
calls `run_from_points2d` directly to show it is exactly the same thing in one call.

## Setup

Import the precise pieces the CLI uses from each module, and point at the data.
`N_FRAMES` controls how much of the 64-frame recording we process &mdash; small
enough to run quickly, large enough for calibration to have something to chew on.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# The exact modules `deeperfly run` composes:
from deeperfly import Config, video, viz
from deeperfly.skeleton import Skeleton
from deeperfly.pose2d import backends, inference
from deeperfly.pose2d.download import download_torch_weights
from deeperfly.triangulate import apply_visibility, triangulate, reprojection_error
from deeperfly.pipeline import calibrate, reconstruct_ransac, run_from_points2d
from deeperfly.correction import smooth_one_euro
from deeperfly.io import PoseResult

# Render plots inline. deeperfly's matplotlib helpers select the headless "Agg"
# backend only when none is set yet (force=False), so importing them in later cells
# won't clobber this inline backend.
%matplotlib inline

# Locate the repo root (so the notebook works regardless of the launch dir).
REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent
DATA_DIR = REPO / "examples" / "data"
CONFIG = REPO / "examples" / "cameras.toml"

# Camera indices 0..6 in the filenames map onto the canonical fly-rig names,
# in the order the example config lists them (right-hind .. left-hind).
CAMERA_NAMES = ["rh", "rm", "rf", "f", "lf", "lm", "lh"]
N_FRAMES = 60  # frames to process (each example video has 64)
FPS = 100.0

n_avail = video.count_frames([DATA_DIR / "camera_0.mp4"])
print(f"repo:    {REPO}")
print(f"data:    {DATA_DIR}  ({n_avail} frames/camera available)")
print(f"cameras: {CAMERA_NAMES}")

## Step 1 &mdash; the camera rig and the skeleton

`Skeleton.fly()` is the rig-independent description of *what* is tracked: 38
points (left `0..18`, right `19..37`), grouped into limbs, joined by bones, with a
per-camera visibility table. `Config.camera_group` builds the rig geometry from the
TOML (it wraps `CameraGroup.from_config`).

The example rig leaves `principal_point_px` unset, so we hand `camera_group` each
camera's `(H, W)` and it places the principal point at the image center &mdash;
exactly what `deeperfly run` does from the footage (these frames are 960&times;480).
The rig file carries no `[pipeline.bundle_adjustment]`, so we take the calibration
options from the packaged run config (`Config.default()`): the leg-only `keypoints`
that drive calibration and the `fixed` / `shared` parameters that anchor the world
gauge.

In [ ]:
skeleton = Skeleton.fly()
config = Config.read(CONFIG)  # the example rig (cameras only)

# The frames are 960x480 and the rig leaves principal_point_px unset, so pass each
# camera's (H, W) and camera_group infers the principal point as the image center.
probe = np.asarray(video.read_frames(DATA_DIR / "camera_0.mp4", stop=1))[0]
H, W = probe.shape[:2]
image_sizes = {name: (H, W) for name in CAMERA_NAMES}
cameras = config.camera_group(image_sizes=image_sizes)

# Bundle-adjustment options, mapped to `calibrate`'s kwargs the same way
# `deeperfly run` does. The rig file carries no [pipeline.bundle_adjustment], so we
# take the packaged run defaults: the leg-only `keypoints` -> `ba_keypoints`, the
# `fixed`/`shared` world gauge, and the scipy least_squares kwargs (max_nfev, loss).
ba = Config.default().bundle_adjustment
calibrate_kwargs = {"fixed": ba.fixed, "shared": ba.shared, **ba.least_squares}
if ba.keypoints is not None:
    calibrate_kwargs["ba_keypoints"] = ba.keypoints

print(
    f"skeleton: {skeleton.name!r}  {skeleton.n_points} points, "
    f"{len(skeleton.bones)} bones, {skeleton.n_limbs} limbs"
)
print(f"limbs: {skeleton.limb_names}")
print(f"frame size (H, W) = {(H, W)};  principal point -> {[(W - 1) / 2, (H - 1) / 2]}")
for c in cameras:
    print(f"  {c.name:>3}: center={np.round(c.position, 1)}  focal={c.intr[0]:.0f}px")

In [ ]:
from deeperfly.cameras import Camera


def plot_camera(camera: Camera, ax, length=None, **kwargs):
    "Draw a camera as an RGB axis triad at its world center"
    if length is None:
        length = np.linalg.norm(camera.tvec) * 0.2
    for axis, color in zip(camera.rmat, "rgb"):
        ax.plot(
            *zip(camera.position, camera.position + axis * length),
            color=color,
            **kwargs,
        )

In [ ]:
# The 7 cameras orbit the world origin (where the fly sits). Plot their centers.
fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(projection="3d")
for c in cameras:
    plot_camera(c, ax, length=10, alpha=0.8)
    ax.text(
        *c.position + np.array([0, 0, 10]),
        s=c.name,
        color="k",
        fontsize=9,
        ha="center",
        va="center",
    )
ax.scatter([0], [0], [0], color="k", marker="x", s=70)
ax.set_title("Camera rig: centers orbit the world origin")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.set_aspect("equal")
ax.view_init(elev=30, azim=-60)
ax.set_zticks([0])
fig.tight_layout()
plt.show()

## Step 2 &mdash; load the 2D detector

The detector is a stacked-hourglass network &mdash; a PyTorch port of the original
DeepFly2D net that loads the released weights directly. We fetch the cached
checkpoint (downloaded on first use) and load it onto the GPU when one is available.

`inference.fly_camera_layout` returns, per camera, which body **side** the 19
detector channels populate and whether the image must be **mirror-flipped** so the
fly faces the trained orientation. Left cameras image the left side (flipped);
right cameras image the right side (un-flipped). The **front camera** is special:
it is assigned side `"both"` &mdash; it is run **twice**, un-flipped to read the
*right* legs and mirror-flipped to read the *left* legs, so the one front image
observes joints on **both** body sides. `inference.expand_passes` turns this
per-camera layout into a flat list of forward *passes* (the front camera becomes
two passes sharing one physical view). That front-camera bridge is what lets
calibration tie the otherwise-disjoint left and right cameras into a single world
frame (see Step 8).

In [ ]:
ckpt = download_torch_weights()  # cached locally after the first download
model = backends.load_detector(ckpt)
sides, flips = inference.fly_camera_layout(CAMERA_NAMES)

# A "both" camera (the front one) is run twice; expand_passes flattens the
# per-camera layout into forward *passes*. `views` maps each pass back to its
# physical camera, so the two front passes share one view.
views, pass_sides, pass_flips = inference.expand_passes(sides, flips)

print(f"loaded detector from {ckpt}")
print(f"running on device: {backends.detector_device(model)}")
print("per-camera layout:")
for n, s, fl in zip(CAMERA_NAMES, sides, flips):
    print(f"  {n:>3}: side={s:<5} flip={fl}")
print(f"\n{len(views)} detector passes (the front camera runs twice):")
for i, vw in enumerate(views):
    print(
        f"  pass {i}: camera {CAMERA_NAMES[vw]:>3}  "
        f"side={pass_sides[i]:<5} flip={pass_flips[i]}"
    )

## Step 3 &mdash; load synchronized frames

`deeperfly run` locates one video or image sequence per camera through each
camera's `input` glob (e.g. `[cameras.rh]` &rarr; `input = "camera_0.mp4"`). This
recording is seven per-camera videos (`camera_0.mp4` &hellip; `camera_6.mp4`, 64
frames each). We decode the first `N_FRAMES` of each with
`deeperfly.video.read_frames` &mdash; the same reader the CLI uses &mdash; and stack
them into a `(V, T, H, W, 3)` array: for frame `t`, the 7 entries `frames[:, t]` are
the synchronized views.

In [ ]:
n_views = len(CAMERA_NAMES)
# Decode the first N_FRAMES of each camera's video and stack the synchronized
# views into (V, T, H, W, 3). `stop=N_FRAMES` reads frames [0, N_FRAMES).
frames = np.stack(
    [
        np.asarray(video.read_frames(DATA_DIR / f"camera_{v}.mp4", stop=N_FRAMES))
        for v in range(n_views)
    ]
)
print("frames:", frames.shape, frames.dtype)

fig, axes = plt.subplots(2, 4, figsize=(12, 4))
for v, ax in enumerate(axes.ravel()):
    if v < n_views:
        ax.imshow(frames[v, 0], cmap="gray")
        ax.set_title(f"camera_{v} = {CAMERA_NAMES[v]} ({sides[v]})", fontsize=8)
    ax.axis("off")
fig.suptitle("Synchronized raw frame 0 across the 7 cameras")
fig.tight_layout()
plt.show()

## Step 4 &mdash; preprocessing

`inference.preprocess` mirror-flips far-side cameras, resizes to the network input
size (256&times;512), and subtracts the training mean. We run it **per pass**, so
the front camera appears twice: once un-flipped (its right-leg pass) and once
mirror-flipped (its left-leg pass). Below, each pass before and after preprocessing
&mdash; note how flipping makes every view face the trained orientation.

In [ ]:
# Preprocess one image per PASS (the front camera -> two passes, second flipped).
preprocessed_frames = np.stack(
    [
        np.asarray(inference.preprocess(frames[views[i], 0], flip=pass_flips[i]))
        for i in range(len(views))
    ]
)
preprocessed_frames_disp = np.clip(
    preprocessed_frames.transpose(0, 2, 3, 1) + inference.MEAN, 0, 1
)
n_pass = len(views)


def pass_label(i):
    """Short name for pass i, e.g. 'f (left, flipped)'."""
    flip = ", flipped" if pass_flips[i] else ""
    return f"{CAMERA_NAMES[views[i]]} ({pass_sides[i]}{flip})"


fig, axes = plt.subplots(4, 4, figsize=(12, 8))
axs_raw = axes[0::2].ravel()
axs_pre = axes[1::2].ravel()
for i, (ax_raw, ax_pre) in enumerate(zip(axs_raw, axs_pre)):
    if i < n_pass:
        ax_raw.set_title(f"{pass_label(i)}, raw", fontsize=8)
        ax_raw.imshow(frames[views[i], 0], cmap="gray")
        ax_pre.set_title(f"{pass_label(i)}, preprocessed", fontsize=8)
        ax_pre.imshow(preprocessed_frames_disp[i], cmap="gray")
    ax_raw.axis("off")
    ax_pre.axis("off")
fig.suptitle(
    "preprocess() per pass: the front camera appears twice (right + flipped left)"
)
fig.tight_layout()
plt.show()

## Step 5 &mdash; heatmaps

`backends.predict_heatmaps` runs the forward pass over every pass, returning one
heatmap per joint (19 single-side channels) at the network's output resolution. The
peak of each heatmap is the predicted joint location. We overlay them onto each
preprocessed pass below.

In [ ]:
heatmaps = backends.predict_heatmaps(model, preprocessed_frames)  # (P, J, Hh, Ww)
print("heatmaps:", heatmaps.shape, "  (passes, joints, Hh, Ww)")

In [ ]:
from deeperfly.viz import limb_colors

In [ ]:
# Color the 19 single-side detector channels by limb: joints 0..18 (lf/lm/lh legs,
# antenna, stripes) -- the per-side ordering the detector emits. limb_colors returns
# RGBA per skeleton point; take one side's channels and drop alpha.
joint_colors = np.asarray(limb_colors(skeleton))[: inference.N_SIDE_JOINTS, :3]

In [ ]:
ext = [
    0,
    preprocessed_frames_disp.shape[2],
    preprocessed_frames_disp.shape[1],
    0,
]  # stretch heatmap onto the 256x512 input
im = np.empty((heatmaps.shape[2], heatmaps.shape[3], 4))
fig, axes = plt.subplots(2, 4, figsize=(12, 4))
for i, ax in enumerate(axes.ravel()):
    if i < n_pass:
        ax.set_facecolor("k")
        ax.imshow(preprocessed_frames_disp[i], cmap="gray", alpha=0.5)
        ax.set_title(pass_label(i), fontsize=8)
        for j in range(heatmaps.shape[1]):
            im[..., :3] = joint_colors[j]
            im[..., 3] = np.clip(heatmaps[i, j] * 2, 0, 1)
            ax.imshow(im, alpha=0.5, extent=ext)

    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle("Stacked-hourglass heatmaps (one set per pass)")
fig.tight_layout()
plt.show()

## Step 6 &mdash; decode peaks and assemble the skeleton

`heatmap_to_points` takes the (sub-pixel-refined) peak of each heatmap &mdash; a
normalized `(x, y)` plus a confidence. `assemble_skeleton` then scatters each
**pass's** 19 single-side detections into the full 38-point skeleton: a right pass
&rarr; `19..37`, a (flipped) left pass &rarr; `0..18` with the flip undone, scaled
back to original pixels. The `views` argument says which physical camera each pass
belongs to, so the front camera's two passes land on the **same row** &mdash;
filling both halves. In the plot below the front camera now carries *both* leg
sets, while every other camera carries one.

In [ ]:
points_norm, conf0 = inference.heatmap_to_points(heatmaps)  # (P, J, 2) in [0,1], (P, J)

In [ ]:
from deeperfly.viz import limb_colors

image_size = [(W, H)] * n_views  # (width, height) per physical camera
pts0, c0 = inference.assemble_skeleton(
    np.asarray(points_norm),
    np.asarray(conf0),
    pass_sides,
    pass_flips,
    image_size,
    views=views,
    n_views=n_views,
)
print("assembled 2D points:", pts0.shape, " confidence:", c0.shape)
front = CAMERA_NAMES.index("f")
print(
    f"front camera fills left half: {np.isfinite(pts0[front, :19]).any()}, "
    f"right half: {np.isfinite(pts0[front, 19:]).any()}"
)

fig, axes = plt.subplots(2, 4, figsize=(12, 4))
colors = limb_colors(skeleton)
for v, ax in enumerate(axes.ravel()):
    if v < n_views:
        ax.set_facecolor("k")
        ax.imshow(frames[v, 0], cmap="gray")
        extra = " - both sides" if v == front else ""
        ax.set_title(f"camera_{v} = {CAMERA_NAMES[v]} ({sides[v]}){extra}", fontsize=8)
        for a, b in skeleton.bones:
            ax.plot(pts0[v, [a, b], 0], pts0[v, [a, b], 1], color=colors[a])
    ax.axis("off")
fig.suptitle("Assembled 2D skeleton (the front camera carries both leg sets)")
fig.tight_layout()
plt.show()

## Step 7 &mdash; detect the whole sequence

`inference.detect_sequence` repeats steps 4&ndash;6 for every frame, giving the full
2D result: `pts2d` of shape `(V, T, 38, 2)` in pixels and `conf` of shape
`(V, T, 38)`. This is the array `deeperfly run` feeds into the geometry pipeline.

In [ ]:
pts2d, conf = inference.detect_sequence(model, frames, sides, flips)
print("pts2d:", pts2d.shape, "  conf:", conf.shape)

fig, (axa, axb) = plt.subplots(1, 2, figsize=(13, 4))
im = axa.imshow(np.nanmean(conf, axis=1), aspect="auto", cmap="viridis", vmin=0, vmax=1)
axa.set_yticks(range(n_views))
axa.set_yticklabels(CAMERA_NAMES)
axa.set_xlabel("joint index  (0..18 left, 19..37 right)")
axa.set_title("Mean detector confidence per (camera, joint)")
fig.colorbar(im, ax=axa, fraction=0.046)

j, v = 23, 1  # RF tarsus tip seen by camera rm
axb.plot(pts2d[v, :, j, 0], label="x")
axb.plot(pts2d[v, :, j, 1], label="y")
axb.set_xlabel("frame")
axb.set_ylabel("pixel")
axb.legend()
axb.set_title(f"{CAMERA_NAMES[v]}: 2D track of {skeleton.joint_names[j]}")
fig.tight_layout()
plt.show()

## Step 8 &mdash; visibility masking

`run_from_points2d` first applies the skeleton's per-camera visibility: each side
camera sees one body side (plus shared midline points), while the **front camera
sees both sides** (the distal joints of the front/middle legs and the antennae).
Observations a camera *cannot* see are set to NaN and never reach triangulation. In
the mask below, the front (`f`) row is the only one with white cells on **both**
halves &mdash; that cross-side visibility is the bridge that co-registers the left
and right cameras during calibration.

In [ ]:
pts2d_vis = apply_visibility(pts2d, skeleton, CAMERA_NAMES)
mask = skeleton.visibility_mask(CAMERA_NAMES)
before = int(np.isfinite(pts2d).all(-1).sum())
after = int(np.isfinite(pts2d_vis).all(-1).sum())
print(f"finite observations: {before} -> {after}  (masked {before - after} unviewable)")

fr = skeleton.visibility["f"]
print(
    f"front camera 'f' sees {fr.size} joints spanning both sides "
    f"(left {int((fr < 19).sum())}, right {int((fr >= 19).sum())})"
)

fig, ax = plt.subplots(figsize=(8, 3))
ax.imshow(mask, aspect="auto", cmap="Greys_r", interpolation="nearest")
ax.set_yticks(range(n_views))
ax.set_yticklabels(CAMERA_NAMES)
ax.set_xlabel("joint index")
ax.set_title("Visibility mask (white = camera sees this joint)")
for i in range(mask.shape[1]):
    ax.axvline(i - 0.5, color="gray", lw=0.5)
ax.set_xticks(
    np.arange(mask.shape[1]),
    labels=skeleton.joint_names,
    rotation=60,
    rotation_mode="anchor",
    ha="right",
)
ax.tick_params(axis="x", which="both", length=0)
fig.tight_layout()
plt.show()

## Step 9 &mdash; calibration (bundle adjustment)

`pipeline.calibrate` treats the **fly itself** as the calibration target: it
flattens the frames into one point cloud and refines the cameras by bundle
adjustment, using detector confidences as per-observation weights, a robust loss,
and a soft bone-length prior. We compare reprojection error before vs after.

In [ ]:
# Reprojection error with the nominal (un-calibrated) cameras ...
pts3d_init = triangulate(cameras, pts2d_vis)
err_init = reprojection_error(cameras, pts3d_init, pts2d_vis)

# ... then refine the rig by fly-as-target bundle adjustment.
cameras_cal, ba_res = calibrate(cameras, pts2d_vis, conf, skeleton, **calibrate_kwargs)
pts3d_cal = triangulate(cameras_cal, pts2d_vis)
err_cal = reprojection_error(cameras_cal, pts3d_cal, pts2d_vis)

fi, fc = np.isfinite(err_init), np.isfinite(err_cal)
print(f"bundle adjustment: {ba_res.nfev} fn evals, final cost {ba_res.cost:.4g}")
print(
    f"median reproj error  before {np.median(err_init[fi]):.2f}px  ->  "
    f"after {np.median(err_cal[fc]):.2f}px"
)

fig, (axa, axb) = plt.subplots(1, 2, figsize=(13, 4))
bins = np.linspace(0, 40, 60)
axa.hist(
    err_init[fi],
    bins=bins,
    alpha=0.6,
    label=f"before (med {np.median(err_init[fi]):.1f})",
)
axa.hist(
    err_cal[fc], bins=bins, alpha=0.6, label=f"after (med {np.median(err_cal[fc]):.1f})"
)
axa.set_xlabel("reprojection error (px)")
axa.set_ylabel("count")
axa.legend()
axa.set_title("Reprojection error: nominal vs calibrated cameras")

shift = np.linalg.norm(
    np.array([c.position for c in cameras_cal])
    - np.array([c.position for c in cameras]),
    axis=1,
)
axb.bar(CAMERA_NAMES, shift, color="tab:purple")
axb.set_ylabel("camera-center shift (world units)")
axb.set_title("How far bundle adjustment moved each camera")
fig.tight_layout()
plt.show()

## Step 10 &mdash; robust triangulation (RANSAC)

`pipeline.reconstruct_ransac` &mdash; the default reconstructor &mdash; builds each
3D point from its *largest set of mutually consistent views* (RANSAC), so a badly
mislocated 2D detection never enters the fit (rather than corrupting it and being
trimmed afterward). Non-inlier observations are set to NaN. The result is the 3D
pose sequence `(T, 38, 3)`.

In [ ]:
pts3d, pts2d_clean, reproj = reconstruct_ransac(
    cameras_cal, pts2d_vis, threshold=15.0, min_inliers=2
)
fin = np.isfinite(reproj)
n_tri = int(np.isfinite(pts3d).all(-1).sum())
print(f"3D points: {pts3d.shape};  triangulated {n_tri} of {pts3d[..., 0].size}")
print(
    f"reproj error (px): median {np.median(reproj[fin]):.2f}  "
    f"p95 {np.percentile(reproj[fin], 95):.2f}"
)

t = N_FRAMES // 2
fig = plt.figure(figsize=(11, 5))
for k, (elev, azim) in enumerate([(20, -60), (85, -90)]):
    ax = fig.add_subplot(1, 2, k + 1, projection="3d")
    viz.plot_skeleton_3d(pts3d[t], skeleton, ax=ax, elev=elev, azim=azim)
    ax.set_title(f"frame {t}  (elev={elev}, azim={azim})", fontsize=9)
    ax.set_aspect("equal")
fig.suptitle("Triangulated 3D pose (RANSAC multi-view consensus)")
fig.tight_layout()
plt.show()

## Step 11 &mdash; temporal smoothing

The last stage applies a NaN-aware 1-Euro filter along time, suppressing jitter
while staying responsive to fast motion.

In [ ]:
pts3d_sm = smooth_one_euro(pts3d, FPS)

j = 23  # RF tarsus tip
fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
for k, lab in enumerate("xyz"):
    axes[k].plot(pts3d[:, j, k], ".-", alpha=0.5, label="raw")
    axes[k].plot(pts3d_sm[:, j, k], "-", lw=2, label="1-Euro")
    axes[k].set_ylabel(lab)
axes[0].legend(loc="upper right")
axes[0].set_title(f"Temporal smoothing of {skeleton.joint_names[j]} (3D position)")
axes[-1].set_xlabel("frame")
fig.tight_layout()
plt.show()

## Step 12 &mdash; the same thing in one call, and save the result

Everything from Step 8 on &mdash; visibility &rarr; calibrate &rarr; reconstruct &rarr;
smooth &mdash; is exactly what `pipeline.run_from_points2d` (and therefore
`deeperfly run`) does internally. We call it directly on the raw `pts2d` / `conf`,
save the `PoseResult` to HDF5, and reload it to confirm the round-trip.

In [ ]:
result = run_from_points2d(
    cameras,
    skeleton,
    pts2d,
    conf,
    do_calibrate=True,
    calibrate_kwargs=calibrate_kwargs,
    triangulation="ransac",
    smooth="one_euro",
    fps=FPS,
    meta={"source": str(DATA_DIR), "n_frames_input": N_FRAMES},
)

out = REPO / "results" / "fly_pose_walkthrough.h5"
out.parent.mkdir(parents=True, exist_ok=True)
result.save(out)

re = result.reproj_error
fr = np.isfinite(re)
print(f"wrote {out}")
print(f"  views x frames x points : {result.pts2d.shape}")
print(f"  3D points               : {result.pts3d.shape}")
print(
    f"  reprojection error (px) : median {np.median(re[fr]):.2f}  "
    f"p95 {np.percentile(re[fr], 95):.2f}"
)

reloaded = PoseResult.load(out)
print(
    f"  reloaded {reloaded.n_views} views x {reloaded.n_frames} frames; "
    f"has 3D = {reloaded.pts3d is not None}"
)

### Sanity check: reproject the calibrated 3D back onto every camera

If calibration and triangulation are consistent, projecting the recovered 3D pose
through the calibrated cameras should land on the fly in each raw view.

In [ ]:
t = 0
proj = result.cameras.project(result.pts3d[t])  # (V, 38, 2)
fig = viz.overlay_grid(
    proj,
    skeleton,
    images=[frames[v, t] for v in range(n_views)],
    camera_names=CAMERA_NAMES,
    ncols=4,
)
fig.suptitle("Calibrated 3D pose reprojected onto every camera (frame 0)")
plt.show()

## Mapping back to the CLI

| Notebook step | CLI / library call |
| --- | --- |
| Steps 1&ndash;2 | `Config.camera_group`, `Skeleton.fly()`, `backends.load_detector` |
| Steps 3&ndash;7 | `inference.fly_camera_layout` + `inference.expand_passes` + `inference.detect_sequence` |
| Steps 8&ndash;12 | `pipeline.run_from_points2d` (visibility &rarr; calibrate &rarr; reconstruct &rarr; smooth &rarr; save) |

`detect_sequence` (and the candidate path enabled by `[pipeline].do_pictorial_structures`)
expands the passes internally, so the front camera is run twice there too &mdash;
the manual cells above just unfold what it does.

`deeperfly run` drives all of this from one config. `deeperfly init` writes a
self-contained `config.toml` (the `[cameras.*]` rig with each camera's `input`
glob, the detector, pipeline, bundle adjustment and the skeleton). The default
cameras already map to this recording's `camera_0.mp4` &hellip; `camera_6.mp4`, so
the whole notebook collapses to:

```bash
deeperfly init config.toml          # then edit [cameras] for your own rig
deeperfly run examples/data -c config.toml -o results/   # detect -> 3D -> overlay videos

deeperfly inspect results/poses.h5                       # inspect the saved result
```

`run` writes `results/poses.h5` (the `PoseResult`) plus one MP4 per
`[[pipeline.visualization.videos]]` entry &mdash; by default `results/pose2d.mp4`
(the raw 2D detections) and `results/pose3d.mp4` (the triangulated skeleton
reprojected into every view). The CLI uses the config's defaults &mdash; RANSAC
triangulation and leg-only bundle adjustment &mdash; matching the 38-point output
of the manual cells above. Merging the left/right abdominal stripes into shared
midline points (if you want it) is left to downstream analysis: average each
`l_stripe*` with its `r_stripe*` counterpart.